Evan Edelstein
EN.605.645.82.SP26

# Module 7 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

# Unification

This is actually Part I of a two part assignment. In a later module, you'll implement a Forward Planner. In order to do that, however, you need to have a unifier. It is important to note that you *only* need to implement a unifier. Although the module talked about resolution, you do not need to implement anything like "standardizing apart". From the unifier's point of view, that should already have been done.

Unification is simply the *syntactic* balancing of expressions. There are only 3 kinds of expressions: constants, lists and (logic) variables. Constants and lists are only equal to each other if they're exactly the same thing or can be made to be the same thing by *binding* a value to a variable.

It really is that simple...expressions must be literally the same (identical) except if one or the other (or both) has a variable in that "spot".

## S-Expressions

With that out of the way, we need a language with which to express our constants, variables and predicates and that language will be based on s-expressions.

**constants** - There are two types of constants, values and predicates. Values should start with an uppercase letter. Fred is a constant value, so is Barney and Food. Predicates are named using lowercase letters. loves is a predicate and so is hates. This is only a convention. Secret: your code does not need to treat these two types of constants differently.

**variables** - these are named using lowercase letters but always start with a question mark. ?x is a variable and so is ?yum. This is not a convention.

**expressions (lists)** - these use the S-expression syntax a la LISP. (loves Fred Wilma) is an expression as is (friend-of Barney Fred) and (loves ?x ?y).

## Parsing

In [35]:
import tokenize
from io import StringIO
from typing import List, Dict, Tuple

This uses the above libraries to build a Lisp structure based on atoms. It is adapted from [simple iterator parser](http://effbot.org/zone/simple-iterator-parser.htm) (deadlink). The first function is the `atom` function.

In [36]:
def atom(next, token):
    if token[1] == "(":
        out = []
        token = next()
        while token[1] != ")":
            out.append(atom(next, token))
            token = next()
            if token[1] == " ":
                token = next()
        return out
    elif token[1] == "?":
        token = next()
        return "?" + token[1]
    else:
        return token[1]

The next function is the actual `parse` function:

In [37]:
def parse(exp):
    src = StringIO(exp).readline
    tokens = tokenize.generate_tokens(src)
    return atom(tokens.__next__, tokens.__next__())

**Note** there was a change between 2.7 and 3.0 that "hid" the next() function in the tokenizer.

From a Python perspective, we want to turn something like "(loves Fred ?x)" to ["loves" "Fred" "?x"] and then work with the second representation as a list of strings. The strings then have the syntactic meaning we gave them previously.

In [38]:
parse("Fred")

'Fred'

In [39]:
parse("?x")

'?x'

In [40]:
parse("(loves Fred ?x)")

['loves', 'Fred', '?x']

In [41]:
parse("(father_of Barney (son_of Barney))")

['father_of', 'Barney', ['son_of', 'Barney']]

## Unifier

Now that that's out of the way, here is the imperative pseudocode for unification. This is a classic recursive program with a number of base cases. Students for some reason don't like it, try the algorithm in the book, can't get it to work and then come back to this pseudocode.

Work through the algorithm by hand with your Self-Check examples if you need to but I'd suggest sticking with this implementation. It does work.

Here is imperative pseudocode for the algorithm:

```
def unification( exp1, exp2):
    # base cases
    if exp1 and exp2 are constants or the empty list:
        if exp1 = exp2 then return {}
        else return FAIL
    if exp1 is a variable:
        if exp1 occurs in exp2 then return FAIL
        else return {exp1/exp2}
    if exp2 is a variable:
        if exp2 occurs in exp1 then return FAIL
        else return {exp2/exp1}

    # inductive step
    first1 = first element of exp1
    first2 = first element of exp2
    result1 = unification( first1, first2)
    if result1 = FAIL then return FAIL
    apply result1 to rest of exp1 and exp2
    result2 = unification( rest of exp1, rest of exp2)
    if result2 = FAIL then return FAIL
    return composition of result1 and result2
```

`unification` can return...

1. `None` (if unification completely fails)
2. `{}` (the empty substitution list) or 
3. a substitution list that has variables as keys and substituted values as values, like {"?x": "Fred"}. 

Note that the middle case sometimes confuses people..."Sam" unifying with "Sam" is not a failure so you return {} because there were no variables so there were no substitutions. You do not need to further resolve variables. If a variable resolves to an expression that contains a variable, you don't need to do the substition.

If you think of a typical database table, there is a column, row and value. This Tuple is a *relation* and in some uses of unification, the "thing" in the first spot..."love" above is called the relation. If you have a table of users with user_id, username and the value then the relation is:

`(login ?user_id ?username)`

*most* of the time, the relation name is specified. But it's not impossible for the relation name to be represented by a variable:

`(?relation 12345 "smooth_operator")`

Your code should handle this case (the pseudocode does handle this case so all  you have to do is not futz with it).

Our type system is very simple. We can get by with just a few boolean functions. The first tests to see if an expression is a variable.

In [42]:
def is_variable(exp):
    return isinstance(exp, str) and exp[0] == "?"

In [43]:
is_variable("Fred")

False

In [44]:
is_variable("?fred")

True

The second tests to see if an expression is a constant:

In [45]:
def is_constant(exp):
    return isinstance(exp, str) and not is_variable(exp)

In [46]:
is_constant("Fred")

True

In [47]:
is_constant("?fred")

False

In [48]:
is_constant(["loves", "Fred", "?wife"])

False

It might also be useful to know that:

<code>
type( "a")
&lt;type 'str'>
type( "a") == str
True
type( "a") == list
False
type( ["a"]) == list
True
</code>


You need to write the `unification` function described above. It should work with two expressions of the type returned by `parse`. See `unify` for how it will be called. It should return the result of unification for the two expressions as detailed above and in the book. It does not have to make all the necessary substitions (for example, if ?y is bound to ?x and 1 is bound to ?y, ?x doesn't have to be replaced everywhere with 1. It's enough to return {"?x":"?y", "?y":1}. For an actual application, you would need to fix this!)

-----

<a id="occurs"></a>
## occurs

*`occurs` Determine if a variable occurs in a S-expression. The expression is traversed recursively.* **Used by**: [unification](#unification)

* **variable** str - variable to search for 
* **expression** List - S-expression to test occurrence of variable in. An expression is represented as a list of strings or sub-expressions

**returns** bool - does variable occur in expression

In [49]:
def occurs(variable, expression):
    if variable == expression:
        return True

    if isinstance(expression, list):  # recurse
        for subexpressions in expression:
            if occurs(variable, subexpressions):
                return True
    return False

In [50]:
assert occurs("?x", ["son", "?x", "Bam_Bam"]) # test 1 - occurs in top level of expression
assert occurs("?y", ["son", "?x", "Bam_Bam"]) is False # test 2 - variable does not occur
assert occurs("?y", ["son", "?x", ["loves", "?y", "Bam_Bam"]]) # test 3 - variable occurs in sub-expression

<a id="apply"></a>
## apply

*`apply` Using a substitution dictionary that maps variables to values, swap any occurrence of variable in expression with its value in the map.* **Used by**: [unification](#unification)

* **substitutions** Dict - a mapping of variables to values to swap with 
* **expression** List - S-expression to replace variables in. An expression is represented as a list of strings or sub-expressions

**returns** List - modified expression 

In [51]:
def apply(substitutions: Dict, expression: List) -> List:
    return [substitutions[i] if is_variable(i) and i in substitutions else i for i in expression]

In [52]:
assert apply({"?x": "Barney"}, ["son", "?x", "Bam_Bam"]) == ["son", "Barney", "Bam_Bam"] # test 1 - properly applies substitution
assert apply({"?y": "Fred"}, ["son", "?x", "Bam_Bam"]) == ["son", "?x", "Bam_Bam"] # test 2 - no modification if variable not in substitution list
assert apply({"?x": "Barney"}, ["son", "?x", ["loves", "?x", "Bam_Bam"]]) == ["son", "Barney", ["loves", "?x", "Bam_Bam"]]  # test 3 - only applies substitution in the top level of the expression

<a id="composition"></a>
## composition

*`composition` Merge two dictionaries. If any keys are shared by the two dictionaries None is returned* **Used by**: [unification](#unification)


* **substitutions1** Dict - dictionary to merge
* **substitutions2** Dict - dictionary to merge 
* **trace** Bool - if true print debug information

**returns** Dict | None - merged dictionary or None if keys overlap

In [53]:
def composition(substitutions1: Dict, substitutions2: Dict, trace: bool = False):
    if any(x in substitutions2 for x in substitutions1):
        return None

    result = {**substitutions1, **substitutions2}
    if trace:
        print(f"New substitution List: {substitutions1} + {substitutions2} -> {result}")
    return result

In [54]:
d1 = {"?x": "Barney"}
d2 = {"?y": "Fred"}
assert composition(d1, d2) == {"?x": "Barney", "?y": "Fred"} # test 1 - proper composition of dictionaries

d1 = {"?x": "Barney"}
d2 = {"?y": "Fred"}
assert composition(d1, d2) == {"?x": "Barney", "?y": "Fred"} and composition(d2, d1) == {"?x": "Barney", "?y": "Fred"}  # test 1.5 - order independent


d1 = {"?x": "Barney"}
d2 = {"?x": "Fred"}
assert composition(d1, d2) == None # test 2 - if any shared keys, None is returned

d1 = {"?x": "Barney"}
d2 = {}
assert composition(d1, d2) == {"?x": "Barney"} and composition(d2, d1) == {"?x": "Barney"} # test 3 - empty dictionary does not overwrite result

<a id="deatomize"></a>
## deatomize

*`deatomize` Recursively transform an expression, represented as list of strings or sub-expressions, into a string with sub expressions parenthesized.* **Used by**: [assign](#assign)

* **expression** List - list representation of an S-expression to convert into a string representation.

**returns** str - string representation of the expression 

In [55]:
def deatomize(expression):
    if isinstance(expression, str): # base-case
        return expression

    if isinstance(expression, list): # recurse
        str_expression = []
        for subexpressions in expression:
            result = deatomize(subexpressions)
            if result is None:
                return None
            str_expression.append(result)
        return "(" + " ".join(str_expression) + ")"

    return None

In [56]:
s_expression1 = ["son", "?y", "Barney"]
assert deatomize(s_expression1) == "(son ?y Barney)"  # test 1 - no nested expressions

s_expression2 = ["son", "?y", ["son", "Barney"]]
assert deatomize(s_expression2) == "(son ?y (son Barney))"  # test 2 - single nested expression

s_expression3 = ["son", "?y", ["son", "Barney", ["love", "Fred"]]]
assert deatomize(s_expression3) == "(son ?y (son Barney (love Fred)))"  # test 3 - more than one nested expression

<a id="assign"></a>
## assign

*`assign` Create a dictionary (substitution list) with variable as the key and expression as the value. If the expression is a list, it is parsed to a string using [deatomize](#deatomize).* **Used by**: [unification](#unification)

* **variable** str - variable to use a key of substitution list
* **expression** List | str - expression or constant to use a value in substitution list. An expression is represented as a list of strings or sub-expressions.

**returns** Dict - dictionary mapping variable to expression

In [57]:
def assign(variable: str, expression: List | str ) -> Dict:
    if isinstance(expression, list):
        expression = deatomize(expression)
    return {variable: expression}

In [58]:
s_expression1 = "Barney"
assert assign("?x", s_expression1) == {"?x": "Barney"}  # test 1 - no nested expressions

s_expression2 = ["son", "?y", ["son", "Barney"]]
assert assign("?x", s_expression2) == {"?x": "(son ?y (son Barney))"}  # test 2 - single nested expression

s_expression3 = ["son", "?y", ["son", "Barney", ["love", "Fred"]]]
assert assign("?x", s_expression3) == {"?x" :  "(son ?y (son Barney (love Fred)))"}  # test 3 - more than one nested expression

<a id="split_expression"></a>
## split_expression

*`split_expression` Split an expression into a tuple pair with the first element of the expression and the rest of the elements of the expression.* **Used by**: [unification](#unification)

* **list_expression** List - S-expression to split, An expression is represented as a list of strings or sub-expressions 

**returns** Tuple[List | str , List | str] - split expression into first and remaining 

In [59]:
def split_expression(list_expression: List) -> Tuple[List | str, List | str]:
    first1, rest1 = list_expression[0], list_expression[1:]
    return first1, rest1

<a id="is_unmodifiable"></a>
## is_unmodifiable

*`is_unmodifiable` Check if two expressions are either both constants or are both empty lists. This means they can no longer be unified.* **Used by**: [unification](#unification)


* **list_expression1** List - S-expression represented as a list of strings or sub-expressions
* **list_expression2** List - S-expression represented as a list of strings or sub-expressions 

**returns** bool - can the expressions be unified further 

In [60]:
def is_unmodifiable(list_expression1: List | str, list_expression2: List | str) -> bool:
    if is_constant(list_expression1) and is_constant(list_expression2):
        return True

    if (isinstance(list_expression1, list) and isinstance(list_expression2, list)) and (len(list_expression1) == 0 and len(list_expression2) == 0):
        return True
    return False

In [61]:
assert is_unmodifiable("Fred" , "Barney") is True # test 1 - constants
assert is_unmodifiable([], []) is True # test 2 - empty lists
assert is_unmodifiable("Fred", []) is False # test 3 - constant and empty list

<a id="unification"></a>
## unification

*`unification` This function evaluates two S-expressions and if possible, returns a substitution list to make both expressions identical. Expressions are represented as lists of strings, or nested sub-expressions. Three base cases are tested at the start of the function. First, the expressions are checked for modifiability [is_unmodifiable](#is_unmodifiable), that is, if the expressions are empty lists or constants (strings). If so,  we can return an empty substitution list if the constants equal one another, or None to indicate unification failed since there is no more substitutions to make. The second set of base cases signify a substitution is possible between the expressions since one has been distilled down to a constant. If we have not reached a base-case we continue by extracting the first element of each expression using [split_expression](#split_expression) and recursively calling unification on them. If the result of the recursive call returns a substitution list, is applied to the remained of each expression by swapping any occurrence of the variable in the substitution list with the corresponding value. We then recursively call unification on the (possibly modified) remainder of each input expression. If this returns a substitution list is is merged with the first recursive calls substitution list ([composition](#composition)) and returned.* **Uses** [is_unmodifiable](#is_unmodifiable), [split_expression](#split_expression) and [composition](#composition)

* **list_expression1** List | str - S-expression to test, an expression is represented as a list of strings or sub-expressions 
* **list_expression2** List | str - S-expression to test, an expression is represented as a list of strings or sub-expressions 
* **trace** bool - if true print debug information 


**returns** Dict | None - a substitution list if a substitution is possible or None if unification fails

In [62]:
def unification(list_expression1: List , list_expression2: List, trace: bool = False) -> Dict | None:
    print(f"Trying to unify {list_expression1} with {list_expression2}") if trace else None
    if is_unmodifiable(list_expression1, list_expression2):
        return {} if list_expression1 == list_expression2 else None

    if is_variable(list_expression1):
        return None if occurs(list_expression1, list_expression2) else assign(list_expression1, list_expression2)

    if is_variable(list_expression2):
        return None if occurs(list_expression2, list_expression1) else assign(list_expression2, list_expression1)

    first1, rest1 = split_expression(list_expression1)
    first2, rest2 = split_expression(list_expression2)
    result1 = unification(first1, first2, trace)
    if result1 is None:
        return None

    rest1, rest2 = apply(result1, rest1), apply(result1, rest2)
    result2 = unification(rest1, rest2, trace)
    return None if result2 is None else composition(result1, result2, trace)

In [63]:
def list_check(parsed_expression):
    if isinstance(parsed_expression, list):
        return parsed_expression
    return [parsed_expression]

The `unification` pseudocode only takes lists so we have to make sure that we only pass a list.
However, this has the side effect of making "foo" unify with ["foo"], at the start.
That's ok.

In [64]:
def unify(s_expression1, s_expression2, trace: bool = False):
    list_expression1 = list_check(parse(s_expression1))
    list_expression2 = list_check(parse(s_expression2))
    return unification(list_expression1, list_expression2, trace)

In [65]:
unify("(future George Fred)", "(future ?y ?y)", trace=True)

Trying to unify ['future', 'George', 'Fred'] with ['future', '?y', '?y']
Trying to unify future with future
Trying to unify ['George', 'Fred'] with ['?y', '?y']
Trying to unify George with ?y
Trying to unify ['Fred'] with ['George']
Trying to unify Fred with George


**Note** If you see the error,

```
tokenize.TokenError: ('EOF in multi-line statement', (2, 0))
```
You most likely have unbalanced parentheses in your s-expression.

## Test Cases

Use the expressions from the Self Check as your test cases...

In [66]:
def run_tests(test_cases):
    for i, case in enumerate(test_cases):
        exp1, exp2, expected, message = case
        actual = unify(exp1, exp2)
        print(f"[{i + 1}] Testing {exp1} & {exp2} - {message}...")
        pre = "PASS" if actual == expected else "FAIL"
        print(f"{pre} actual = {actual} | expected = {expected}")
        print("\n")
        assert expected == actual

In [67]:
self_check_test_cases = [
    ["(son Barney Barney)", "(daughter Wilma Pebbles)", None, "non-equal constants"],
    ["(Fred)", "(Barney)", None, "self check case - invalid unification of non-equal constant"],
    ["(Pebbles)", "(Pebbles)", {}, "self check case - unification of the same constant returns an empty substitution list"],
    ["(quarry_worker Fred)", "(quarry_worker ?x)", {"?x": "Fred"}, "self check case - valid unification of single variable"],
    ["(son Barney ?x)", "(son ?y Bam_Bam)", {"?y": "Barney", "?x": "Bam_Bam"}, "self check case - valid unification of two variables"],
    ["(married ?x ?y)", "(married Barney Wilma)", {"?x": "Barney", "?y": "Wilma"}, "self check case - valid unification of two variable within the same expression"],
    ["(son Barney ?x)", "(son ?y (son Barney))", {"?y": "Barney", "?x": "(son Barney)"}, "self check case - valid assignment of a variable to an expression"],
    ["(son Barney ?x)", "(son ?y (son ?y))", {"?y": "Barney", "?x": "(son ?y)"}, "self check case - valid assignment of a variable to an expression with replacement"],
    ["(son Barney Bam_Bam)", "(son ?y (son Barney))", None, "self check case -  invalid unification of a constant function to a constant"],
    ["(loves Fred Fred)", "(loves ?x ?x)", {"?x": "Fred"}, "self check case - valid substitution of the same constant to the same variable"],
    ["(future George Fred)", "(future ?y ?y)", None, "self check case - invalid substitution of the different constants to the same variable"],
]
run_tests(self_check_test_cases)

[1] Testing (son Barney Barney) & (daughter Wilma Pebbles) - non-equal constants...
PASS actual = None | expected = None


[2] Testing (Fred) & (Barney) - self check case - invalid unification of non-equal constant...
PASS actual = None | expected = None


[3] Testing (Pebbles) & (Pebbles) - self check case - unification of the same constant returns an empty substitution list...
PASS actual = {} | expected = {}


[4] Testing (quarry_worker Fred) & (quarry_worker ?x) - self check case - valid unification of single variable...
PASS actual = {'?x': 'Fred'} | expected = {'?x': 'Fred'}


[5] Testing (son Barney ?x) & (son ?y Bam_Bam) - self check case - valid unification of two variables...
PASS actual = {'?y': 'Barney', '?x': 'Bam_Bam'} | expected = {'?y': 'Barney', '?x': 'Bam_Bam'}


[6] Testing (married ?x ?y) & (married Barney Wilma) - self check case - valid unification of two variable within the same expression...
PASS actual = {'?x': 'Barney', '?y': 'Wilma'} | expected = {'?x': 'Barn

Now add at least **five (5)** additional test cases of your own making, explaining exactly what you are testing. They should not be testing the same things as the self check test cases above.

In [ ]:
test_cases = [
    ["(?x )", "(?y)", {"?x": "?y"}, "unification of variables to each other"],
    ["()", "()", {}, "unification of two empty lists"],
    ["(loves ?x)", "(loves (son ?x))", None, "invalid unification of expression containing itself"],
    ["(loves (son ?x) Bam_Bam)", "(loves (son Fred) Bam_Bam)", {"?x": "Fred"}, "unification of nested subexpressions"],
    ["(?x Fred Wilma)", "(loves Fred Wilma)", {"?x": "loves"}, "unification of predicates"],
]

run_tests(test_cases)

[1] Testing (?x ) & (?y) - unification of variables to each other...
PASS actual = {'?x': '?y'} | expected = {'?x': '?y'}


[2] Testing () & () - unification of two empty lists...
PASS actual = {} | expected = {}


[3] Testing (loves ?x) & (loves (son ?x)) - invalid unification of expression containing itself...
PASS actual = None | expected = None


[4] Testing (loves (son ?x) Bam_Bam) & (loves (son Fred) Bam_Bam) - nested subexpressions unify...
PASS actual = {'?x': 'Fred'} | expected = {'?x': 'Fred'}


[5] Testing (?x Fred Wilma) & (loves Fred Wilma) - predicates can unify...
PASS actual = {'?x': 'loves'} | expected = {'?x': 'loves'}




## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.